# USGS: grilla ShakeMap recortada a Cali

Análisis exploratorio independiente de bronce. Fecha de revisión: 13 de noviembre de 2026.

Se examinan los archivos locales originales, no una tabla Delta ejecutada en Databricks.
No se limpia ni sobrescribe la fuente. Las transformaciones que aparezcan se hacen en copias
para evaluar lo que necesitará plata. Los resultados no prueban causalidad ni certifican seguridad.

Cada notebook sigue el orden: procedencia y unidad, calidad por columna y geometría,
distribuciones, límites y reglas para plata. No se muestran registros individuales de edificios oficiales.

In [ ]:
from pathlib import Path
import sys, json, re, unicodedata
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "CLAUDE.md").exists() and (p / "data/externos").is_dir())
sys.path.insert(0, str(ROOT / "analisis_bronce"))
from soporte import (ficha, huella, leer_geojson, mostrar_perfil, geometria, conteos,
                     faltantes, guardar_resumen, grafico_barras, SALIDAS, CRS_METROS)
pd.set_option("display.max_columns", 24)
pd.set_option("display.max_rows", 60)
plt.rcParams.update({"font.size": 11, "figure.dpi": 110})

## 1. Procedencia y unidad

Una fila es un nodo de la grilla de movimiento del suelo del evento us6000tjl2, recortada a Cali. El CSV se deriva de grid.xml, no es una medición individual por edificio.

Este archivo fue materializado por `scripts/descargar_externos.py`. Se distingue de los originales remotos: el script recorta, combina o deriva columnas antes de escribirlo.

In [ ]:
PATH=ROOT/'data/externos/usgs_shakemap_cali.csv'
meta=ficha(PATH)
df=pd.read_csv(PATH,keep_default_na=False)
calidad=mostrar_perfil(df)
x=df.copy()
g=gpd.GeoDataFrame(x,geometry=gpd.points_from_xy(pd.to_numeric(x.lon),pd.to_numeric(x.lat)),crs=4326)
geo=geometria(g)
display(pd.Series({**meta, "filas":len(df),"columnas":len(df.columns)}).to_frame("valor"))
display(pd.Series(geo).to_frame("geometria"))
resumen={"dataset":'15_usgs_shakemap_cali',**meta,"filas":len(df),"columnas":list(df),"calidad_columnas":calidad.to_dict("records"),"geometria":geo}

## 2. Rangos, unidades y resolución

In [ ]:
campos=["mmi","pga","pgv","psa03","psa06","psa10","psa30","svel"]
display(df[campos].describe(percentiles=[.01,.5,.95,.99]))
from scipy.spatial import cKDTree
metrica=g.to_crs(CRS_METROS)
xy=np.column_stack([metrica.geometry.x,metrica.geometry.y])
espaciado=cKDTree(xy).query(xy,k=2)[0][:,1]
qa={"duplicados_lon_lat":int(df.duplicated(["lon","lat"]).sum()),
    "mmi_fuera_1_12":int((~df.mmi.between(1,12)).sum()),
    "pga_negativa":int(df.pga.lt(0).sum()),"pgv_negativa":int(df.pgv.lt(0).sum()),
    "espaciado_mediano_m":float(np.median(espaciado))}
display(pd.Series(qa).to_frame("control"))
unidades=pd.DataFrame([("mmi","intensidad MMI"),("pga","%g"),("pgv","cm/s"),
                       ("psa03/06/10/30","%g"),("svel","m/s")],columns=["campo","unidad_en_grid_xml"])
display(unidades)
fig,ax=plt.subplots(figsize=(8,6));g.plot(ax=ax,column="mmi",legend=True,markersize=22,cmap="YlOrRd")
ax.set(title="MMI modelada en nodos ShakeMap de Cali",xlabel="Longitud",ylabel="Latitud")
fig.tight_layout();fig.savefig(SALIDAS/"15_mmi.png",dpi=150);plt.show()
resumen.update(qa);resumen["rangos"]=df[campos].agg(["min","max"]).to_dict()

## Reglas decididas para plata

- **Seleccionar como sacudida local del evento**, con la versión y el XML original conservados. Puede apoyar el contraste entre zonas; no prueba la causa del daño.
- Convertir PGA/PSA de %g a g en columnas nuevas: dividir por 100. PGV conserva cm/s. Guardar el metadato de unidades del XML junto al CSV.
- Asignar nodo en un CRS métrico y guardar distancia y resolución. Un dato de grilla no es precisión instrumental por predio.
- La grilla ya utiliza información de sitio y observaciones de estaciones/DYFI. No llamar validación independiente a comparar el modelo contra esos mismos insumos ni amplificar de nuevo sin justificar posible doble conteo del suelo.
- Para otra ciudad o terremoto hace falta un extracto y versión comparable, no reutilizar esta grilla de Cali.

In [ ]:
assert huella(PATH) == meta["sha256"], "La fuente cruda cambió durante el análisis"
guardar_resumen('15_usgs_shakemap_cali', resumen)
print("Resumen y gráficos guardados; la fuente cruda conserva su SHA-256.")